# Tutorial 3: Multi-Camera 3D Triangulation

**Pipeline Stage:** Fusing matched 2D poses into 3D skeletons

---

## Overview

This tutorial covers the **third and final stage** of the multi-camera human tracking
pipeline: taking the matched 2D poses and **triangulating** them into 3D skeleton
coordinates.

### Why Triangulation Comes LAST

Triangulation requires two inputs:
1. **2D keypoints per camera, already identity-matched** (from Tutorial 2: ReID)
2. **Camera calibration** describing where each camera is and how it projects 3D to 2D

Without knowing which detection in Camera 1 is the same person as which detection in
Camera 3 (the ReID step), triangulation would try to combine keypoints from *different
people* across cameras, producing nonsensical 3D skeletons. That is why this notebook
reads Tutorial 2's tracked output rather than Tutorial 1's raw per-camera detections
directly.

```
The problem without ReID:
  CAM1: Person A at (100, 200)     CAM3: Person X at (300, 150)
  CAM1: Person B at (500, 300)     CAM3: Person Y at (450, 250)
                                   CAM3: Person Z at (100, 400)

  Which pairs do we triangulate?
  A+X? A+Y? A+Z? B+X? B+Y? B+Z?  <- ReID tells us!
```

### What You Will Learn

1. **Installation** - `sleap-anipose`, `sleap-io`, `h5py`, and dependencies
2. **Understanding the Input Data** - tracked `.slp` files from Tutorial 2, skeleton carried over from Tutorial 1
3. **Building Track-Aligned Arrays** - turning per-camera `.slp` files into the array shape triangulation needs
4. **Camera Calibration** - what a calibration TOML file contains, and matching its camera names to your data
5. **Running Triangulation** - using `sleap-anipose`, parallelized across people
6. **Inspecting 3D Results** - loading and understanding the output H5 file
7. **3D Visualization** - plotting skeletons in 3D with matplotlib
8. **Scene Normalization** - orienting the coordinate system so Z = up, floor = Z=0 (vectorized, not a per-frame loop)

### Pipeline Context

```
+---------------------+     +----------------------+     +---------------------+
|  Tutorial 1          | --> |  Tutorial 2           | --> |  Tutorial 3 (HERE)   |
|  YOLO Pose (2D)     |     |  Person ReID          |     |  3D Triangulation    |
|  per-camera          |     |  cross-camera match   |     |  multi-camera fusion |
+---------------------+     +----------------------+     +---------------------+
       outputs:                    outputs:                    outputs:
    2D keypoints per cam      reid_<CAM>.slp (who=who)      3D skeleton coords
```

### Prerequisites

- Completed Tutorial 1 (2D pose results, `pose_results/*_pose.slp` per camera)
- Completed Tutorial 2 (tracked `reid_results/reid_<CAM>.slp` per camera, proofread)
- Camera calibration file (`.toml`) for your multi-camera rig
- All camera videos synchronized (same assumption Tutorial 2 depends on)

---

## Part 1: Installation

| Package | Purpose | Install |
|---|---|---|
| `sleap-anipose` | Multi-camera triangulation engine | `pip install sleap-anipose` |
| `sleap-io` | Reading Tutorial 2's tracked `.slp` files | `pip install sleap-io` |
| `h5py` | Read/write HDF5 pose data files | `pip install h5py` |
| `joblib` | Optional: parallelize triangulation across people | `pip install joblib` |
| `matplotlib` | 3D visualization | `pip install matplotlib` |
| `seaborn` | Color palettes for plots | `pip install seaborn` |
| `numpy` | Array operations | `pip install numpy` |
| `pandas` | Tabular data inspection | `pip install pandas` |

### About sleap-anipose

`sleap-anipose` bridges SLEAP pose estimation with the Anipose triangulation library.
It handles:
- Applying camera calibration parameters
- Running Direct Linear Transformation (DLT) triangulation
- Optional spatiotemporal smoothing and limb-length constraints

### How Triangulation Works (Conceptually)

```
Camera 1 sees nose at pixel (320, 240)
Camera 2 sees nose at pixel (510, 195)
Camera 3 sees nose at pixel (280, 310)
         |
         v  (using calibration: intrinsics + extrinsics)
Cast a ray from each camera through its 2D detection
         |
         v
Find the 3D point that best satisfies all rays
         |
         v
nose_3d = (1.42, 0.85, 1.73) meters
```

In [ ]:
# ============================================================
# STEP 1: Install required packages
# ============================================================
# Uncomment and run if not yet installed:

# !pip install sleap-anipose sleap-io h5py joblib matplotlib seaborn numpy pandas

In [ ]:
# ============================================================
# STEP 2: Verify installation
# ============================================================
import h5py
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sleap_io as sio

print(f"h5py:       {h5py.__version__}")
print(f"numpy:      {np.__version__}")
print(f"pandas:     {pd.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"sleap-io:   {sio.__version__}")

try:
    import sleap_anipose
    print(f"sleap-anipose: available")
except ImportError:
    print("WARNING: sleap-anipose not installed. Install with: pip install sleap-anipose")

try:
    import joblib
    print(f"joblib:     {joblib.__version__}  (parallel triangulation across people available)")
except ImportError:
    print("joblib:     NOT installed -- STEP 5 will fall back to running one person at a time.")

---

## Part 2: Understanding the Inputs

Triangulation needs two things, both already produced by the earlier tutorials:

### From Tutorial 2: Tracked `.slp` Files (the actual input)

Tutorial 2's deliverable, `reid_results/reid_<CAM>.slp`, is a **tracked** SLEAP file per
camera: every detection that got a confident identity carries a `Track` object named
`person_0` ... `person_{N_PEOPLE-1}`, and, this is the property triangulation depends on,
the same name means the same human in every camera's file.

This notebook reads those files directly with `sleap-io`:

```python
labels = sio.load_slp(path)
for lf in labels:
    for inst in lf.instances:
        if inst.track is not None:
            pts = inst.numpy()        # (n_nodes, 2)
```

and builds a dense, track-aligned array from them, no `.analysis.h5` export needed. (An
earlier draft of this notebook expected a SLEAP `.analysis.h5` per camera; that format
requires a *fixed* instance count for the whole video and only exists after tracking has
already been finalized in the SLEAP GUI. Reading the `.slp` directly works with Tutorial 2's
output as-is and skips an unnecessary export/import round trip.)

### From Tutorial 1 (indirectly): The Skeleton

The 17-node COCO skeleton originates in Tutorial 1 and is carried through Tutorial 2's
output unchanged. Rather than hardcoding a fourth copy of the node names in this notebook,
STEP 3 reads them straight from `labels.skeletons[0]`. If the skeleton ever changes
upstream, this notebook picks it up automatically instead of silently drifting out of sync.

### File Organization

```
my_experiment/
+-- reid_results/                  <- REID_DIR   (Tutorial 2 output)
|   +-- reid_CAM1.slp
|   +-- reid_CAM2.slp
|   +-- ...
+-- calibration.toml                <- CALIB_PATH
```

### A Note on Camera Names

`calibration.toml` stores whatever camera names were used during calibration capture,
which are not guaranteed to match Tutorial 2's `CAMERA_REGEX` labels exactly (`CAM1` vs
`cam1` is a common mismatch). STEP 5 matches them case- and punctuation-insensitively and
fails loudly, rather than silently, if a camera can't be matched.

In [ ]:
# ============================================================
# STEP 3: Configuration and camera discovery  <<<<<  EDIT THE PATHS
# ============================================================
from pathlib import Path

import numpy as np
import pandas as pd
import sleap_io as sio

# =============================================================
#  1. YOUR DATA  <<<<< change these two lines
# =============================================================
REID_DIR = Path("PUT_YOUR_TUTORIAL_2_REID_RESULTS_FOLDER_HERE")   # Tutorial 2's OUT_DIR
CALIB_PATH = Path("PUT_PATH_TO_calibration.toml_HERE")

# A filled-in example, for reference:
#   REID_DIR   = Path("/data/my_experiment/reid_results")
#   CALIB_PATH = Path("/data/my_experiment/calibration.toml")

OUT_DIR = Path("triangulation_results")   # all output lands here, next to this notebook

# =============================================================
#  2. FILE NAMING  (defaults match Tutorial 2's output)
# =============================================================
REID_GLOB = "reid_*.slp"     # Tutorial 2 writes "reid_<CAM>.slp"
REID_PREFIX = "reid_"        # stripped off the filename to get the camera label

# =============================================================
#  Nothing below here needs editing.
# =============================================================
if not REID_DIR.exists():
    raise FileNotFoundError(
        f"REID_DIR does not exist:\n    {REID_DIR}\n\n"
        "Edit section 1 at the top of this cell. If you have not run Tutorial 2 yet, "
        "do that first -- it produces the reid_<CAM>.slp files this notebook needs."
    )
if not CALIB_PATH.exists():
    raise FileNotFoundError(
        f"CALIB_PATH does not exist:\n    {CALIB_PATH}\n\n"
        "Point this at your camera rig's calibration.toml (see Part 3 below for the format)."
    )

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Discover cameras from Tutorial 2's tracked .slp files ───
POSE_PATHS = {}
for p in sorted(REID_DIR.glob(REID_GLOB)):
    cam = p.stem[len(REID_PREFIX):] if p.stem.startswith(REID_PREFIX) else p.stem
    POSE_PATHS[cam] = p

CAMERAS = list(POSE_PATHS)
if not CAMERAS:
    raise FileNotFoundError(
        f"No files matching {REID_GLOB!r} found under REID_DIR:\n    {REID_DIR}\n\n"
        "Check that Tutorial 2's STEP 13 actually wrote its output here."
    )

# ── Load every camera's tracked Labels once; everything below reuses these ──
LABELS = {cam: sio.load_slp(str(path)) for cam, path in POSE_PATHS.items()}

skeleton = LABELS[CAMERAS[0]].skeletons[0]
NODE_NAMES = [n.name for n in skeleton.nodes]
EDGE_IDX = [(NODE_NAMES.index(e.source.name), NODE_NAMES.index(e.destination.name))
            for e in skeleton.edges]

print("=" * 78)
print("CONFIGURATION")
print("=" * 78)
print(f"  REID_DIR    {REID_DIR}")
print(f"  CALIB_PATH  {CALIB_PATH}")
print(f"  OUT_DIR     {OUT_DIR.resolve()}")

print(f"\n{'camera':<8}{'frames':>8}{'tracks':>8}   file")
summary_rows = []
for cam in CAMERAS:
    L = LABELS[cam]
    track_names = [t.name for t in L.tracks]
    print(f"{cam:<8}{len(L):>8}{len(L.tracks):>8}   {POSE_PATHS[cam].name}")
    summary_rows.append({"Camera": cam, "Frames": len(L), "Tracks": len(L.tracks),
                          "Track names": track_names})

print(f"\nSkeleton: {len(NODE_NAMES)} nodes -- {NODE_NAMES}")

# ── Every camera must agree on the track names, or "person_2" would mean a ──
# ── different child in different files, which would silently corrupt triangulation.
all_track_names = [tuple(r["Track names"]) for r in summary_rows]
if len(set(all_track_names)) > 1:
    print("\nWARNING: cameras do not all have the same track names:")
    for r in summary_rows:
        print(f"  {r['Camera']}: {r['Track names']}")
    raise ValueError(
        "Track names must match exactly across all cameras (this is what makes "
        "'person_2' mean the same person in every file). Re-run Tutorial 2 Stage 3 "
        "if these have diverged, or check you pointed REID_DIR at one consistent run."
    )
TRACK_NAMES = list(all_track_names[0])
N_PEOPLE = len(TRACK_NAMES)
print(f"\n{N_PEOPLE} consistent tracks across {len(CAMERAS)} cameras: {TRACK_NAMES}")

pd.DataFrame(summary_rows)[["Camera", "Frames", "Tracks"]]

---

## Part 3: Camera Calibration

### What is Camera Calibration?

To triangulate 2D points from multiple cameras into 3D, you need to know:

1. **Intrinsic parameters** - focal length, principal point, distortion (per camera)
   - These describe the camera's internal optics
   - How a 3D point maps to a pixel location

2. **Extrinsic parameters** - rotation and translation (per camera)
   - These describe where the camera is in the world
   - Position and orientation of each camera

### Calibration TOML Format

```toml
[cam1]
  name = "CAM1"                                     # must match a discovered camera (Part 2)
  size = [1920, 1080]                              # image resolution
  matrix = [[fx, 0, cx], [0, fy, cy], [0, 0, 1]]  # intrinsic matrix
  distortions = [k1, k2, p1, p2, k3]               # lens distortion
  rotation = [r1, r2, r3]                           # Rodrigues rotation
  translation = [tx, ty, tz]                        # camera position
```

### How to Create a Calibration File

- Use a **checkerboard** or **ChArUco board** visible to all cameras simultaneously
- Tools: OpenCV `calibrateCamera()`, Anipose calibration, or Caltech toolbox
- The calibration maps each camera name to its intrinsic and extrinsic parameters

In [ ]:
# ============================================================
# STEP 4: Inspect the calibration file
# ============================================================
if CALIB_PATH.exists():
    with open(CALIB_PATH, "r") as f:
        content = f.read()
    print("Calibration file contents (first 2000 chars):")
    print(content[:2000])
    if len(content) > 2000:
        print(f"\n... ({len(content)} total characters)")
else:
    print(f"Calibration file not found at: {CALIB_PATH}")
    print("You need to create a calibration file for your camera rig.")

---

## Part 4: Running Triangulation with sleap-anipose

### The Process

1. Build a `(n_cams, n_frames, n_tracks, n_nodes, 2)` array from Tutorial 2's tracked `.slp`
   files, one array slot per person (this is where the identity map already baked into the
   track names does its job -- no separate lookup step needed)
2. Match discovered cameras to the calibration's camera order
3. Triangulate all people in one call; each person's bundle adjustment is independent, so we
   parallelize across people rather than running them one at a time

### Key Parameters

| Parameter | Description | Typical Value |
|---|---|---|
| `p2d` | 2D points array or path to a session directory | built in STEP 5 |
| `calib` | Path to calibration TOML | `CALIB_PATH` |
| `scale_smooth` | Smoothing weight (higher = smoother) | 2 |
| `scale_length` | Limb length constraint weight | 2 |
| `n_deriv_smooth` | Order of derivative for smoothing | 2 |

In [ ]:
# ============================================================
# STEP 5: Build track-aligned 2D arrays and run triangulation
# ============================================================
import re

import h5py
import sleap_anipose as slap
from aniposelib.cameras import CameraGroup

PARALLEL_TRACKS = True   # fan out one process per person -- each person's bundle
                         # adjustment is fully independent, so this is close to a
                         # free N_PEOPLE-way speedup on a multi-core machine.
N_JOBS = -1              # passed to joblib; -1 = use all cores

TRIANGULATE_KWARGS = dict(
    scale_smooth=2,
    scale_length=2,
    scale_length_weak=2,
    n_deriv_smooth=2,
)

cgroup = CameraGroup.load(str(CALIB_PATH))
cam_order = cgroup.get_names()


# ── Match discovered cameras to the calibration's camera names ──────
# Calibration files store whatever name was used during calibration capture,
# which is not guaranteed to match Tutorial 2's CAMERA_REGEX labels byte for
# byte -- "CAM1" here vs "cam1" in calibration.toml is a common real-world snag.
def _norm(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())


norm_to_cam = {_norm(c): c for c in CAMERAS}
calib_to_discovered = {}
missing = []
for calib_name in cam_order:
    match = norm_to_cam.get(_norm(calib_name))
    if match is None:
        missing.append(calib_name)
    else:
        calib_to_discovered[calib_name] = match

if missing:
    raise ValueError(
        f"calibration.toml has camera(s) {missing} with no matching reid_<CAM>.slp "
        f"under REID_DIR.\n  Discovered cameras: {CAMERAS}\n  Calibration cameras: {cam_order}\n"
        "Camera names must correspond (case/punctuation-insensitive) -- rename one "
        "side to match, or adjust CAMERA_REGEX in Tutorial 2 and re-export."
    )

print(f"Calibration camera order: {cam_order}")
print("Matched to discovered cameras: "
      + ", ".join(f"{c}->{calib_to_discovered[c]}" for c in cam_order))


def labels_to_track_array(labels, track_names, n_nodes):
    """(n_frames, n_tracks, n_nodes, 2) array, tracks in the exact given order.

    Built by hand in one pass over instances rather than via Labels.numpy():
    that method silently falls back to an "untracked, arbitrary order" layout
    whenever a file never has more than one instance in the same frame, which
    would break the fixed person_k -> column k assumption every camera here
    depends on. This is O(total detections), not O(frames x tracks).
    """
    name_to_col = {name: i for i, name in enumerate(track_names)}
    n_tracks = len(track_names)

    video = labels.videos[0]
    last_frame = max((lf.frame_idx for lf in labels), default=-1)
    n_frames_from_video = len(video)
    if n_frames_from_video > 0:
        last_frame = max(last_frame, n_frames_from_video - 1)
    elif last_frame < 0:
        return np.zeros((0, n_tracks, n_nodes, 2), dtype="float32")
    else:
        print(f"    (video file not reachable from this machine -- using the last "
              f"labeled frame ({last_frame}) as the frame count, which under-counts "
              f"if the tail of the video has no detections)")

    arr = np.full((last_frame + 1, n_tracks, n_nodes, 2), np.nan, dtype="float32")
    for lf in labels:
        for inst in lf.instances:
            if inst.track is None:
                continue
            col = name_to_col.get(inst.track.name)
            if col is None:
                continue
            arr[lf.frame_idx, col] = inst.numpy()
    return arr


n_nodes = len(NODE_NAMES)
print("\nBuilding per-camera track-aligned arrays...")
cam_arrays = {}
for cam in CAMERAS:
    cam_arrays[cam] = labels_to_track_array(LABELS[cam], TRACK_NAMES, n_nodes)
    print(f"  {cam}: {cam_arrays[cam].shape}")

# All cameras must agree on frame count to stack -- "synchronized videos" is
# Tutorial 2's stated assumption, but truncate defensively rather than crash
# on an off-by-a-few-frames mismatch.
n_frames_per_cam = {cam: cam_arrays[cam].shape[0] for cam in CAMERAS}
n_frames = min(n_frames_per_cam.values())
if len(set(n_frames_per_cam.values())) > 1:
    print(f"\nWARNING: cameras report different frame counts: {n_frames_per_cam}")
    print(f"  Truncating all cameras to the shortest: {n_frames} frames.")

points_2d = np.stack(
    [cam_arrays[calib_to_discovered[c]][:n_frames] for c in cam_order],
    axis=0,
)  # (n_cams, n_frames, n_tracks, n_nodes, 2)
print(f"\nStacked 2D points: {points_2d.shape}  "
      f"(cams, frames, tracks, nodes, xy) in calibration camera order {cam_order}")

OUT_H5 = OUT_DIR / "points3d.h5"


def _triangulate_one_track(track_points):
    """track_points: (n_cams, n_frames, n_nodes, 2) for a single person."""
    return cgroup.triangulate_optim(track_points, **TRIANGULATE_KWARGS)


print(f"\nTriangulating {N_PEOPLE} tracks x {n_frames:,} frames "
      f"({'parallel' if PARALLEL_TRACKS else 'sequential'})...")

if PARALLEL_TRACKS:
    try:
        from joblib import Parallel, delayed
        results = Parallel(n_jobs=N_JOBS, backend="loky")(
            delayed(_triangulate_one_track)(points_2d[:, :, t]) for t in range(N_PEOPLE)
        )
    except ImportError:
        print("  joblib not installed (pip install joblib) -- falling back to sequential.")
        results = [_triangulate_one_track(points_2d[:, :, t]) for t in range(N_PEOPLE)]
else:
    results = [_triangulate_one_track(points_2d[:, :, t]) for t in range(N_PEOPLE)]

points_3d = np.stack(results, axis=1)  # (n_frames, n_tracks, n_nodes, 3)

with h5py.File(OUT_H5, "w") as f:
    f.create_dataset("tracks", data=points_3d, chunks=True,
                      compression="gzip", compression_opts=1)
    f["tracks"].attrs["Description"] = (
        f"Shape: (n_frames, n_tracks, n_nodes, 3). Tracks in order: {TRACK_NAMES}. "
        f"Cameras in calibration order: {cam_order}."
    )
    f.create_dataset("track_names", data=np.array(TRACK_NAMES, dtype="S"))
    f.create_dataset("node_names", data=np.array(NODE_NAMES, dtype="S"))

print(f"\nDone. 3D points: {points_3d.shape}")
print(f"Saved: {OUT_H5}")

---

## Part 5: Inspecting the 3D Results

After triangulation, we get an H5 file with 3D skeleton coordinates.

### Output H5 Structure

```
points3d.h5
+-- tracks       shape: (n_frames, n_tracks, n_nodes, 3)
|                       - n_tracks = matched individuals (from Tutorial 2's track names)
|                       - n_nodes = 17 COCO keypoints
|                       - 3 = (x, y, z) coordinates
|                       - NaN for joints that couldn't be triangulated
+-- track_names  the person_k names, in tracks-axis order
+-- node_names   the skeleton's node names, in nodes-axis order
```

In [ ]:
# ============================================================
# STEP 6: Load and inspect the 3D triangulated data
# ============================================================
with h5py.File(OUT_H5, "r") as f:
    print("H5 File Keys:")
    for key in f.keys():
        ds = f[key]
        print(f"  {key}: shape={ds.shape}, dtype={ds.dtype}")
    data = f["tracks"][()]

num_frames, num_tracks, num_joints, num_dims = data.shape

print(f"\nData shape: {data.shape}")
print(f"  Frames:     {num_frames}")
print(f"  Tracks:     {num_tracks}  ({TRACK_NAMES})")
print(f"  Joints:     {num_joints}  ({NODE_NAMES})")
print(f"  Dimensions: {num_dims} (x, y, z)")

# Data quality
nan_count = np.isnan(data).sum()
total = np.prod(data.shape)
print(f"\nData quality:")
print(f"  NaN values: {nan_count:,} / {total:,} ({nan_count/total*100:.1f}%)")
print(f"  Valid data: {total - nan_count:,} ({(total-nan_count)/total*100:.1f}%)")

print(f"\nSample -- Frame 0, {TRACK_NAMES[0]}:")
for i, name in enumerate(NODE_NAMES):
    x, y, z = data[0, 0, i]
    status = "valid" if not np.isnan(x) else "NaN"
    print(f"  {name:<18}: ({x:>8.2f}, {y:>8.2f}, {z:>8.2f})  [{status}]")

---

## Part 6: 3D Visualization

Let's plot the 3D skeletons using matplotlib's 3D projection.

### COCO Skeleton Connections

```
Face:  nose-eyes, eyes-ears
Arms:  shoulder-elbow-wrist
Torso: shoulder-shoulder, shoulder-hip, hip-hip
Legs:  hip-knee-ankle
```

Connections are read from the skeleton loaded in STEP 3 (`EDGE_IDX`) rather than
hardcoded again here, so this stays correct if the skeleton ever changes upstream.

In [ ]:
# ============================================================
# STEP 7: Plot a single 3D frame with skeleton connections
# ============================================================
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

skeleton_connections = EDGE_IDX   # from the skeleton loaded in STEP 3

track_colors = ["red", "blue", "green", "purple", "orange", "brown", "teal", "gold"]


def plot_3d_frame(data, frame_idx, ax=None):
    """Plot 3D skeletons for all tracks in a given frame."""
    if ax is None:
        fig = plt.figure(figsize=(12, 10))
        ax = fig.add_subplot(111, projection="3d")

    frame_data = data[frame_idx]

    for track_idx in range(frame_data.shape[0]):
        kpts = frame_data[track_idx]  # (17, 3)
        color = track_colors[track_idx % len(track_colors)]
        label = TRACK_NAMES[track_idx] if track_idx < len(TRACK_NAMES) else f"track_{track_idx}"

        if np.isnan(kpts).all():
            continue

        valid = ~np.isnan(kpts[:, 0])
        ax.scatter(kpts[valid, 0], kpts[valid, 1], kpts[valid, 2],
                   color=color, s=50, label=label)

        for j1, j2 in skeleton_connections:
            if valid[j1] and valid[j2]:
                ax.plot([kpts[j1, 0], kpts[j2, 0]],
                        [kpts[j1, 1], kpts[j2, 1]],
                        [kpts[j1, 2], kpts[j2, 2]],
                        color=color, linewidth=2)

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(f"Frame {frame_idx}")
    ax.legend()
    return ax


# Plot frame 0
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")
plot_3d_frame(data, 0, ax)
plt.tight_layout()
plt.savefig(OUT_DIR / "3d_skeleton_frame.png", dpi=150)
plt.show()
print(f"Saved: {OUT_DIR / '3d_skeleton_frame.png'}")

In [ ]:
# ============================================================
# STEP 8: Plot multiple frames to see motion over time
# ============================================================
sample_frames = np.linspace(0, num_frames - 1, 4, dtype=int)

fig, axes = plt.subplots(1, 4, figsize=(24, 6),
                          subplot_kw={"projection": "3d"})

for ax, fidx in zip(axes, sample_frames):
    plot_3d_frame(data, fidx, ax)

plt.suptitle("3D Skeletons Over Time", fontsize=16)
plt.tight_layout()
plt.savefig(OUT_DIR / "3d_skeleton_sequence.png", dpi=150)
plt.show()
print(f"Saved: {OUT_DIR / '3d_skeleton_sequence.png'}")

---

## Part 7: Scene Normalization

Raw triangulated coordinates are in an arbitrary coordinate system defined by
the camera calibration. For analysis, we want:

- **Z axis = up** (perpendicular to the floor)
- **Floor at Z = 0**
- **X-Y plane = ground plane**

### Strategy

1. **Find "up"**: Average foot-to-nose vectors across all frames/tracks
2. **Build rotation matrix**: Align this direction with the Z axis (Rodrigues' formula)
3. **Find floor level**: Use average ankle Z position after rotation
4. **Translate**: Shift so floor is at Z = 0

The frame-by-frame version of this loops over every `(frame, track)` pair in pure
Python. For a full multi-hour session that loop runs millions of times; STEP 9 below
does the same math as array-wide numpy operations instead.

In [ ]:
# ============================================================
# STEP 9: Normalize the 3D scene - Z = up, floor at Z = 0
# ============================================================
# Vectorized over the whole (frames, tracks, nodes, 3) array instead of a
# Python double for-loop over every (frame, track) pair.
import seaborn as sns
sns.set_theme(style="ticks")

NOSE = NODE_NAMES.index("nose")
LEFT_ANKLE = NODE_NAMES.index("left_ankle")
RIGHT_ANKLE = NODE_NAMES.index("right_ankle")

nose = data[:, :, NOSE, :]        # (F, T, 3)
l_ankle = data[:, :, LEFT_ANKLE, :]
r_ankle = data[:, :, RIGHT_ANKLE, :]

l_valid = ~np.isnan(l_ankle).any(axis=-1)
r_valid = ~np.isnan(r_ankle).any(axis=-1)
nose_valid = ~np.isnan(nose).any(axis=-1)

# --- Step 1: "up" direction, same fallback order as the frame-by-frame version ---
# (both ankles visible -> average; else whichever one is; else this sample is unusable)
both = l_valid & r_valid
feet = np.where(both[..., None], (l_ankle + r_ankle) / 2,
                np.where(l_valid[..., None], l_ankle, r_ankle))
feet_valid = l_valid | r_valid

up = nose - feet
up_norm = np.linalg.norm(up, axis=-1)
usable = feet_valid & nose_valid & (up_norm > 0.1)

up_vectors = up[usable] / up_norm[usable, None]
avg_up = up_vectors.mean(axis=0)
avg_up = avg_up / np.linalg.norm(avg_up)
print(f'Average "up" direction (raw): {avg_up}  (from {len(up_vectors):,} frame/track samples)')

# --- Step 2: build rotation to align 'up' with Z axis (Rodrigues' formula) ---
target_up = np.array([0, 0, 1])
rotation_axis = np.cross(avg_up, target_up)
rotation_axis_norm = np.linalg.norm(rotation_axis)

if rotation_axis_norm < 1e-6:
    R = np.eye(3) if np.dot(avg_up, target_up) > 0 else -np.eye(3)
    angle = 0.0
else:
    rotation_axis = rotation_axis / rotation_axis_norm
    cos_angle = np.clip(np.dot(avg_up, target_up), -1, 1)
    angle = np.arccos(cos_angle)
    K = np.array([
        [0, -rotation_axis[2], rotation_axis[1]],
        [rotation_axis[2], 0, -rotation_axis[0]],
        [-rotation_axis[1], rotation_axis[0], 0]
    ])
    R = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * K @ K

print(f"Rotation angle: {np.degrees(angle):.1f} degrees")

# --- Step 3: apply the rotation to every valid point at once ---
valid_pt = ~np.isnan(data[..., 0])              # (F, T, J)
normalized_data = np.full_like(data, np.nan)
normalized_data[valid_pt] = data[valid_pt] @ R.T

# --- Step 4: floor level, vectorized nanpercentile over both ankles' rotated Z ---
ankle_z = normalized_data[:, :, [LEFT_ANKLE, RIGHT_ANKLE], 2]   # (F, T, 2)
floor_z = np.nanpercentile(ankle_z, 5)
normalized_data[:, :, :, 2] -= floor_z

print(f"Floor level (before shift): {floor_z:.3f}")
print(f"After normalization: Z=0 is the floor, Z>0 is up")

In [ ]:
# ============================================================
# STEP 10: Visualize the normalized 3D scene
# ============================================================
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")

frame_idx = 0
frame_data = normalized_data[frame_idx]
colors = sns.color_palette("muted", n_colors=num_tracks)

for track_idx in range(num_tracks):
    kpts = frame_data[track_idx]
    valid = ~np.isnan(kpts[:, 0])
    if not valid.any():
        continue

    color = colors[track_idx]
    label = TRACK_NAMES[track_idx] if track_idx < len(TRACK_NAMES) else f"track_{track_idx}"
    ax.scatter(kpts[valid, 0], kpts[valid, 1], kpts[valid, 2],
               color=color, s=60, label=label)

    for j1, j2 in skeleton_connections:
        if valid[j1] and valid[j2]:
            ax.plot([kpts[j1, 0], kpts[j2, 0]],
                    [kpts[j1, 1], kpts[j2, 1]],
                    [kpts[j1, 2], kpts[j2, 2]],
                    color=color, linewidth=2.5)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z (up)")
ax.set_title(f"Normalized 3D Scene - Frame {frame_idx}\n(Z=0 is floor, Z>0 is up)")
ax.legend()

plt.tight_layout()
plt.savefig(OUT_DIR / "normalized_3d_scene.png", dpi=150)
plt.show()
print(f"Saved: {OUT_DIR / 'normalized_3d_scene.png'}")

---

## Summary - Full Pipeline

### The Complete Pipeline (Correct Order)

```
Raw Video (6 synchronized cameras)
    |
    v  Tutorial 1: YOLO Pose Estimation
2D Poses per camera (.slp)
  - Detects people and 17 keypoints per camera independently
  - Handles landscape and portrait cameras
    |
    v  Tutorial 2: Person Re-Identification
Tracked 2D Poses per camera (reid_<CAM>.slp)
  - Crops people/torsos from each camera
  - Extracts OSNet appearance embeddings
  - Matches same person across cameras per frame
  - MUST happen before triangulation!
    |
    v  Tutorial 3: 3D Triangulation (this tutorial)
3D Skeleton Data (points3d.h5)
  - Reads Tutorial 2's tracked .slp directly (no .analysis.h5 export needed)
  - Triangulates matched 2D keypoints into 3D, parallelized across people
  - Normalizes scene (Z=up, floor=0), vectorized across all frames/tracks
    |
    v
Consistent 3D Tracking with Identity
  - Each person has a consistent ID across all cameras
  - Full 3D skeleton trajectory over time
  - Ready for downstream analysis (gait, behavior, etc.)
```

### Key Takeaways

- **Order matters**: Pose -> ReID -> Triangulation (not Pose -> Triangulation -> ReID)
- Triangulation quality depends on **calibration accuracy**, **ReID accuracy**, and **pose consistency**
- NaN values indicate joints that couldn't be reliably triangulated (occluded in too many views)
- Scene normalization is essential for downstream analysis

### Possible Extensions

- Temporal smoothing of 3D trajectories beyond what `scale_smooth` already applies
- Iterative refinement: use 3D positions (epipolar constraints) to improve ReID matches
- Chunked/windowed triangulation for very long sessions, trading a little smoothing
  continuity at chunk boundaries for bounded memory and incremental checkpointing
- Real-time pipeline by processing frames as they arrive